# Segmentação de clientes de shopping (K-Means)

Dataset clássico de clustering do Kaggle (*Mall Customer Segmentation
Data*): 200 clientes de um shopping, com idade, renda anual e um
"spending score" (índice de 1 a 100 atribuído pelo próprio shopping com
base em comportamento e padrão de gasto).

**Pergunta de negócio:** dá pra agrupar esses clientes em segmentos com
comportamento parecido, de um jeito que o time de marketing consiga agir
em cima disso (campanha diferente por segmento, por exemplo)?

Diferente dos outros dois notebooks (que são supervisionados - tem um
alvo pra prever), aqui é aprendizado não-supervisionado: não existe
"resposta certa", o critério de qualidade é a métrica de cluster
(silhueta) e principalmente se os grupos fazem sentido de negócio.

Dataset: `data/mall_customers.csv`.

## Importando bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_style("whitegrid")
RANDOM_STATE = 42

## Carregando os dados

In [ ]:
df = pd.read_csv("data/mall_customers.csv")
df = df.rename(columns={"Annual Income (k$)": "AnnualIncome", "Spending Score (1-100)": "SpendingScore"})
print(df.shape)
df.describe()

## Análise exploratória

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
sns.histplot(df["Age"], bins=20, ax=axes[0])
axes[0].set_title("Distribuição de idade")
sns.histplot(df["AnnualIncome"], bins=20, ax=axes[1])
axes[1].set_title("Distribuição de renda anual (k$)")
sns.histplot(df["SpendingScore"], bins=20, ax=axes[2])
axes[2].set_title("Distribuição de spending score")
plt.tight_layout()
plt.show()

In [ ]:
sns.scatterplot(data=df, x="AnnualIncome", y="SpendingScore", hue="Gender")
plt.title("Renda anual x Spending score")
plt.show()

Só de olhar o scatter acima já dá pra ver grupos visuais (esse dataset é
famoso justamente por ilustrar isso tão bem: cantos com renda alta/gasto
baixo, renda alta/gasto alto etc.) - a pergunta é quantos clusters
capturam essa estrutura sem "cortar" um grupo real em dois.

## Preparando os dados

In [ ]:
features = ["Age", "AnnualIncome", "SpendingScore"]
X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Escolhendo o número de clusters

Duas heurísticas complementares: método do cotovelo (inércia - soma das
distâncias quadráticas ao centróide) e silhouette score (o quão bem
separado cada ponto está do cluster vizinho mais próximo, de -1 a 1).
O cotovelo sozinho é subjetivo; olhar os dois junto ajuda a decidir com
menos viés.

In [ ]:
inertias = []
silhouettes = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(list(k_range), inertias, marker="o")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inércia")
axes[0].set_title("Método do cotovelo")

axes[1].plot(list(k_range), silhouettes, marker="o", color="darkorange")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette por k")

plt.tight_layout()
plt.show()

In [ ]:
best_k = k_range[np.argmax(silhouettes)]
print(f"k com maior silhouette score: {best_k}")

## Modelo final e perfil dos segmentos

In [ ]:
kmeans_final = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
df["Cluster"] = kmeans_final.fit_predict(X_scaled)

cluster_profile = df.groupby("Cluster")[features].mean().round(1)
cluster_profile["n_clientes"] = df["Cluster"].value_counts().sort_index()
cluster_profile

In [ ]:
plt.figure(figsize=(7, 5.5))
sns.scatterplot(data=df, x="AnnualIncome", y="SpendingScore", hue="Cluster", palette="tab10", s=60)
plt.title(f"Segmentos de clientes (k={best_k})")
plt.show()

## Traduzindo cluster em decisão de negócio

Números de cluster (`0`, `1`, `2`...) não significam nada pra quem vai
usar a análise - o último passo é sempre nomear os segmentos com uma
lógica de negócio, olhando a tabela de perfil acima (renda x gasto x
idade média de cada grupo). Nesse dataset, os grupos que tipicamente
aparecem são:

- **Renda alta / gasto alto** - clientes prioritários, maior potencial de
  receita, alvo natural de programa de fidelidade.
- **Renda alta / gasto baixo** - segmento interessante para investigar:
  têm capacidade de gasto mas não estão convertendo, pode valer campanha
  direcionada.
- **Renda baixa / gasto alto** - sensíveis a promoção, engajados mas com
  ticket menor.
- **Renda baixa / gasto baixo** e **grupo médio/central** - geralmente o
  maior volume de clientes, comportamento mais "padrão".

O código acima generaliza para qualquer `k` escolhido pela silhueta; a
etapa de nomear os segmentos é deliberadamente manual, porque exige
contexto de negócio que o algoritmo não tem.